# Experiment Analysis — Span Classification for Decoder-Only Transformer Models


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
import seaborn as sns
import warnings
import os
from pathlib import Path
from matplotlib.gridspec import GridSpec
from IPython.display import Image, display

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})
sns.set_style('whitegrid')

BASE = Path('../Experiment_results')
PLOTS = BASE / 'Plots'
PLOTS.mkdir(exist_ok=True)

MODEL_COLORS = {
    'Gemma-3 4B':   '#4C72B0',
    'Qwen3 8B':     '#DD8452',
    'GPT-OSS 20B':  '#55A868',
    'Llama-3.1 8B': '#C44E52',
}
PROC_COLORS = {
    'Unconstrained':  '#E15759',
    'Whole-Sequence': '#4E79A7',
    'Token-Aware':    '#F28E2B',
}
PROMPT_COLORS = {
    'Base':     '#4C72B0',
    'Markdown': '#DD8452',
    'Short':    '#55A868',
}
print("Setup complete. Plots directory:", PLOTS)


In [ ]:
MODEL_MAP = {
    'google/gemma-3-4b-it': 'Gemma-3 4B', 'gemma3:4b': 'Gemma-3 4B',
    'Qwen/Qwen3-8B': 'Qwen3 8B', 'qwen3:8b': 'Qwen3 8B',
    'meta-llama/Llama-3.1-8B-Instruct': 'Llama-3.1 8B', 'llama3.1:8b': 'Llama-3.1 8B',
    'gpt-oss:20b': 'GPT-OSS 20B',
}
PROMPT_MAP = {
    'SYSTEM_PROMPT_CONTEXT': 'Base', 'SYSTEM_PROMPT_CONTEXT_MD': 'Markdown',
    'SYSTEM_PROMPT_CONTEXT_MD_SHORT': 'Short',
    'SYSTEM_PROMPT_TOXIC_SPANS': 'Base', 'SYSTEM_PROMPT_TOXIC_SPANS_MD': 'Markdown',
    'SYSTEM_PROMPT_TOXIC_SPANS_MD_SHORT': 'Short',
    'SYSTEM_PROMPT_LEGALQA': 'Base', 'SYSTEM_PROMPT_LEGALQA_MD': 'Markdown',
    'SYSTEM_PROMPT_LEGALQA_MD_SHORT': 'Short',
}


def config_label(row):
    em = str(row.get('eval_mode', '')).lower()
    if 'unconstrained' in em:
        return 'Unconstrained'
    pc = str(row.get('processor_class', '')).lower()
    if 'token_aware' in pc:
        return 'Token-Aware'
    return 'Whole-Sequence'

def safe_err(val):
    return 0.0 if (val is None or (isinstance(val, float) and np.isnan(val))) else float(val)

def normalize_constrained(df, char_level=False):
    df = df.copy()
    p = 'char_' if char_level else ''
    has_pct = f'{p}f1_pct' in df.columns or 'f1_pct' in df.columns
    if not has_pct:
        for c in ['precision', 'recall', 'f1', 'accuracy']:
            if c in df.columns:
                df[f'{c}_pct'] = df[c] * 100
        if char_level:
            for c in ['char_precision', 'char_recall', 'char_f1']:
                if c in df.columns:
                    df[f'{c}_pct'] = df[c] * 100
        for c in ['wrong_text_rate', 'unaligned_entity_rate', 'all_entities_wrongly_unaligned_rate']:
            col = c + '_avg'
            if col in df.columns:
                df[f'{c}_pct'] = df[col] * 100
            elif c in df.columns:
                df[f'{c}_pct'] = df[c] * 100
        for s in ['f1_std_pct', 'precision_std_pct', 'recall_std_pct', 'accuracy_std_pct',
                  'char_f1_std_pct', 'char_precision_std_pct', 'char_recall_std_pct']:
            if s not in df.columns:
                df[s] = np.nan
    if 'wrong_text_rate_pct' not in df.columns:
        for alt in ['wrong_text_rate_avg', 'wrong_text_rate']:
            if alt in df.columns:
                df['wrong_text_rate_pct'] = df[alt] * 100
                break
    if 'unaligned_entity_rate_pct' not in df.columns:
        for alt in ['unaligned_entity_rate_avg', 'unaligned_entity_rate']:
            if alt in df.columns:
                df['unaligned_entity_rate_pct'] = df[alt] * 100
                break
    df['model_short'] = df['model'].map(MODEL_MAP).fillna(df['model'])
    df['config'] = df.apply(config_label, axis=1)
    if 'batch_size' not in df.columns:
        df['batch_size'] = 1
    main_f1 = f'{p}f1_pct' if f'{p}f1_pct' in df.columns else 'f1_pct'
    df['main_f1_pct'] = df[main_f1] if main_f1 in df.columns else np.nan
    main_std = f'{p}f1_std_pct' if f'{p}f1_std_pct' in df.columns else 'f1_std_pct'
    df['main_f1_std_pct'] = df[main_std].apply(safe_err) if main_std in df.columns else 0.0
    return df

def normalize_context(df, char_level=False):
    df = df.copy()
    p = 'char_' if char_level else ''
    has_pct = f'{p}f1_pct' in df.columns
    if not has_pct:
        for c in ['precision', 'recall', 'f1']:
            col = f'{p}{c}'
            if col in df.columns:
                df[f'{col}_pct'] = df[col] * 100
        for c in ['context_not_in_input_rate', 'entity_not_in_context_rate',
                  'exact_match_rate', 'fuzzy_helped_rate']:
            col_avg = f'{c}_avg'
            if col_avg in df.columns:
                df[f'{c}_pct'] = df[col_avg] * 100
            elif c in df.columns:
                df[f'{c}_pct'] = df[c] * 100
        for s in [f'{p}f1_std_pct', f'{p}precision_std_pct', f'{p}recall_std_pct']:
            if s not in df.columns:
                df[s] = np.nan
        if 'elapsed_minute' in df.columns and 'elapsed_minute_avg' not in df.columns:
            df['elapsed_minute_avg'] = df['elapsed_minute']
    if 'batch_size' not in df.columns:
        df['batch_size'] = 1
    df['model_short'] = df['model'].map(MODEL_MAP).fillna(df['model'])
    if 'system_prompt' in df.columns:
        df['prompt_short'] = df['system_prompt'].map(PROMPT_MAP).fillna(df['system_prompt'])
    main_f1 = f'{p}f1_pct' if f'{p}f1_pct' in df.columns else 'f1_pct'
    df['main_f1_pct'] = df[main_f1] if main_f1 in df.columns else np.nan
    main_std = f'{p}f1_std_pct' if f'{p}f1_std_pct' in df.columns else 'f1_std_pct'
    df['main_f1_std_pct'] = df[main_std].apply(safe_err) if main_std in df.columns else 0.0
    return df

def load_cg(paths, char_level=False):
    frames = []
    for p in paths:
        if Path(p).exists():
            frames.append(normalize_constrained(pd.read_csv(p), char_level=char_level))
        else:
            print(f"WARNING: {p} not found")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_ctx(paths, char_level=False):
    frames = []
    for p in paths:
        if Path(p).exists():
            frames.append(normalize_context(pd.read_csv(p), char_level=char_level))
        else:
            print(f"WARNING: {p} not found")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def save_fig(fig, name):
    path = PLOTS / f'{name}.png'
    fig.savefig(path, dpi=200, bbox_inches='tight')
    print(f'Saved: {path.name}')
    plt.show()
    plt.close(fig)

print("Helper functions loaded.")


## 1. Token/Character Position-Based Approach — Failure Analysis

The token/character position-based approach was the first candidate explored. It attempts to locate entity spans by matching predicted start/end token positions against the source text. This section documents the four key failure modes observed during preliminary experiments.


In [ ]:
import os

screenshot_dir = Path('../screenshots')
screenshots = {
    'example_of_looping.png': 'Failure Mode 1: Non-terminating reasoning loop',
    'json_tokens_correct.png': 'Tokenization with JSON: correct',
    'no_json_tokens_correct.png': 'Tokenization without JSON: inconsistent',
    'big_input_context_json_correct.png': 'Large input: correct JSON context handling',
}

available = {k: v for k, v in screenshots.items() if (screenshot_dir / k).exists()}
if available:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, (fname, title) in zip(axes, available.items()):
        img = mpimg.imread(str(screenshot_dir / fname))
        ax.imshow(img)
        ax.set_title(title, fontsize=10, wrap=True)
        ax.axis('off')
    plt.suptitle('Token/Character Position-Based Approach — Failure Modes', fontsize=13, y=1.02)
    plt.tight_layout()
    save_fig(fig, 'position_based_failures')
else:
    print("Screenshots not found — skipping display.")


**Key failure modes of the token/character position-based approach:**

1. **Non-terminating reasoning loops** — decoder-only models with chain-of-thought prompting sometimes produce unbounded reasoning traces that never emit a structured answer.
2. **Tokenization inconsistency** — the same surface string tokenises differently inside vs. outside a JSON template, making position offsets unreliable.
3. **Off-by-one boundary errors** — start/end token indices predicted by the model do not consistently align with word boundaries in the detokenised text.
4. **Context window pressure** — long documents push the predicted positions beyond the reliable attention range of smaller models.

These failure modes motivate the two alternative approaches evaluated in subsequent sections: the *context-based* approach (Section 2) and the *constrained generation* approach (Section 3).


## 2. CoNLL-2003: Context-Based Approach

The context-based approach extracts named entities as (entity, label, context) triplets. The context string anchors each entity to a unique passage in the source text, avoiding direct reliance on token positions. We evaluate four models across four batch sizes (1, 5, 10, 64) and three prompt variants.


In [ ]:
ctx_exact = load_ctx([
    BASE / 'CoNLL/Context-Based/Csv' / f'ner_document_context_{bs}_BATCHSZ_robust_prompt.csv'
    for bs in [1, 5, 10, 64]
])
ctx_fuzzy = load_ctx([
    BASE / 'CoNLL/Context-Based/Csv' / f'ner_document_context_fuzzy_{bs}_BATCHSZ_robust_prompt.csv'
    for bs in [1, 5, 10, 64]
])
print(f"Exact rows: {len(ctx_exact)}, Fuzzy rows: {len(ctx_fuzzy)}")
print("Batch sizes:", sorted(ctx_exact['batch_size'].unique()))
print("Models:", sorted(ctx_exact['model_short'].unique()))


In [ ]:
ctx_models = ['Gemma-3 4B', 'Qwen3 8B', 'GPT-OSS 20B', 'Llama-3.1 8B']
batch_sizes = [1, 5, 10, 64]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

for idx, model in enumerate(ctx_models):
    ax = axes[idx]
    for match_type, df, ls, label in [
        ('exact', ctx_exact, '-', 'Exact'),
        ('fuzzy', ctx_fuzzy, '--', 'Fuzzy'),
    ]:
        sub = df[df['model_short'] == model].copy()
        agg = sub.groupby('batch_size').agg(
            f1_mean=('main_f1_pct', 'mean'),
            f1_std=('main_f1_pct', 'std'),
            f1_std_col=('main_f1_std_pct', 'mean'),
        ).reindex(batch_sizes)

        f1s = agg['f1_mean'].values
        stds = agg['f1_std_col'].fillna(agg['f1_std'].fillna(0)).values

        color = MODEL_COLORS.get(model, 'gray')
        ax.plot(batch_sizes, f1s, marker='o', linestyle=ls, color=color, linewidth=2, label=label)
        for i, bs in enumerate(batch_sizes):
            if bs < 64 and not np.isnan(f1s[i]) and stds[i] > 0:
                ax.errorbar(bs, f1s[i], yerr=stds[i], fmt='none', color=color,
                            capsize=4, alpha=0.6)

    ax.set_title(model, fontweight='bold')
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('F1 (%)')
    ax.set_xticks(batch_sizes)
    ax.set_xticklabels(['1', '5', '10', '64'])
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend()

plt.suptitle('CoNLL-2003 Context-Based: F1 vs Batch Size (per model)', fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, 'conll_ctx_f1_vs_batch_size_per_model')


In [ ]:
bs1_exact = ctx_exact[ctx_exact['batch_size'] == 1].copy()
prompts = ['Base', 'Markdown', 'Short']

x = np.arange(len(ctx_models))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 5))

for i, prompt in enumerate(prompts):
    sub = bs1_exact[bs1_exact['prompt_short'] == prompt]
    f1s, errs = [], []
    for model in ctx_models:
        row = sub[sub['model_short'] == model]
        f1s.append(row['main_f1_pct'].mean() if len(row) > 0 else np.nan)
        errs.append(safe_err(row['main_f1_std_pct'].mean()) if len(row) > 0 else 0.0)
    bars = ax.bar(x + i * width, f1s, width, label=prompt,
                  color=list(PROMPT_COLORS.values())[i], alpha=0.85,
                  yerr=errs, capsize=4, error_kw={'alpha': 0.7})

ax.set_xticks(x + width)
ax.set_xticklabels(ctx_models)
ax.set_ylabel('F1 (%)')
ax.set_title('CoNLL-2003 Context-Based: Prompt Variant Comparison (BS=1)', fontsize=13)
ax.legend(title='Prompt Variant')
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'conll_ctx_prompt_comparison')


In [ ]:
ctx_stable = ctx_exact[ctx_exact['batch_size'].isin([1, 5, 10])].copy()

metrics = {
    'Precision': 'precision_pct' if 'precision_pct' in ctx_stable.columns else 'main_f1_pct',
    'Recall': 'recall_pct' if 'recall_pct' in ctx_stable.columns else 'main_f1_pct',
    'F1': 'main_f1_pct',
}
if 'precision_pct' not in ctx_stable.columns and 'f1_pct' in ctx_stable.columns:
    metrics['Precision'] = 'precision_pct' if 'precision_pct' in ctx_stable.columns else 'f1_pct'
    metrics['Recall'] = 'recall_pct' if 'recall_pct' in ctx_stable.columns else 'f1_pct'

# Build per-model aggregation
rows = []
for model in ctx_models:
    sub = ctx_stable[ctx_stable['model_short'] == model]
    for label, col in metrics.items():
        if col in sub.columns:
            rows.append({'Model': model, 'Metric': label,
                         'Mean': sub[col].mean(), 'Std': sub[col].std()})
df_metrics = pd.DataFrame(rows)

x = np.arange(len(ctx_models))
metric_list = list(metrics.keys())
width = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(11, 5))
for i, metric in enumerate(metric_list):
    sub = df_metrics[df_metrics['Metric'] == metric]
    vals = [sub[sub['Model'] == m]['Mean'].values[0] if len(sub[sub['Model'] == m]) > 0 else 0
            for m in ctx_models]
    stds = [sub[sub['Model'] == m]['Std'].values[0] if len(sub[sub['Model'] == m]) > 0 else 0
            for m in ctx_models]
    ax.bar(x + i * width, vals, width, label=metric, color=colors[i], alpha=0.85,
           yerr=stds, capsize=4, error_kw={'alpha': 0.7})

ax.set_xticks(x + width)
ax.set_xticklabels(ctx_models)
ax.set_ylabel('Score (%)')
ax.set_title('CoNLL-2003 Context-Based: Average Metrics per Model (BS=1–10)', fontsize=13)
ax.legend(title='Metric')
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'conll_ctx_avg_metrics_per_model')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

diffs, labels_x = [], []
for bs in [1, 5, 10, 64]:
    for model in ctx_models:
        sub_e = ctx_exact[(ctx_exact['batch_size'] == bs) & (ctx_exact['model_short'] == model)]
        sub_f = ctx_fuzzy[(ctx_fuzzy['batch_size'] == bs) & (ctx_fuzzy['model_short'] == model)]
        if len(sub_e) > 0 and len(sub_f) > 0:
            diff = sub_f['main_f1_pct'].mean() - sub_e['main_f1_pct'].mean()
            diffs.append(diff)
            labels_x.append(f'{model}\nBS{bs}')

colors_bar = [MODEL_COLORS.get(lbl.split('\n')[0], 'gray') for lbl in labels_x]
bars = ax.bar(range(len(diffs)), diffs, color=colors_bar, alpha=0.8, edgecolor='white')
ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.7)
ax.set_xticks(range(len(labels_x)))
ax.set_xticklabels(labels_x, fontsize=8)
ax.set_ylabel('Fuzzy F1 − Exact F1 (%)')
ax.set_title('CoNLL-2003 Context-Based: Fuzzy Matching Gain over Exact Matching', fontsize=13)
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)

legend_patches = [mpatches.Patch(color=c, label=m) for m, c in MODEL_COLORS.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
plt.tight_layout()
save_fig(fig, 'conll_ctx_fuzzy_vs_exact_diff')


In [ ]:
bs1 = ctx_exact[ctx_exact['batch_size'] == 1].copy()
diag_cols = {'Context not in input': 'context_not_in_input_rate_pct',
             'Entity not in context': 'entity_not_in_context_rate_pct'}

x = np.arange(len(ctx_models))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))

for i, (label, col) in enumerate(diag_cols.items()):
    vals = []
    for model in ctx_models:
        sub = bs1[bs1['model_short'] == model]
        vals.append(sub[col].mean() if (col in sub.columns and len(sub) > 0) else 0.0)
    ax.bar(x + i * width, vals, width, label=label,
           color=['#4C72B0', '#DD8452'][i], alpha=0.85)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(ctx_models)
ax.set_ylabel('Rate (%)')
ax.set_title('CoNLL-2003 Context-Based: Entity Alignment Diagnostics (BS=1)', fontsize=13)
ax.legend()
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'conll_ctx_alignment_diagnostics')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for model in ctx_models:
    sub = ctx_exact[ctx_exact['model_short'] == model]
    agg = sub.groupby('batch_size')['elapsed_minute_avg'].mean().reindex(batch_sizes)
    ax.plot(batch_sizes, agg.values, marker='o', linewidth=2,
            color=MODEL_COLORS.get(model, 'gray'), label=model)

ax.set_xticks(batch_sizes)
ax.set_xticklabels(['1', '5', '10', '64'])
ax.set_xlabel('Batch Size')
ax.set_ylabel('Runtime (minutes)')
ax.set_title('CoNLL-2003 Context-Based: Runtime vs Batch Size', fontsize=13)
ax.legend()
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'conll_ctx_runtime')


## 3. CoNLL-2003: Constrained Generation Approach

The constrained generation approach uses logits processors to restrict the model's output vocabulary at each decoding step, ensuring that generated spans are substrings of the source document. We compare two processor variants — *whole-sequence* and *token-aware* — against an unconstrained baseline.


In [ ]:
cg = load_cg([
    BASE / 'CoNLL/Constrained-Gen/Csv' / f'hf_all_configs_eval_{bs}_BS_conll2003.csv'
    for bs in [1, 5, 10, 64]
])
print(f"Constrained-gen rows: {len(cg)}")
print("Models:", sorted(cg['model_short'].unique()))
print("Configs:", sorted(cg['config'].unique()))
print("Batch sizes:", sorted(cg['batch_size'].unique()))
cg_models = ['Gemma-3 4B', 'Qwen3 8B', 'Llama-3.1 8B']


In [ ]:
def make_wrong_text_plot(df, models, title, figname, col='wrong_text_rate_pct'):
    df_greedy = df[df.get('do_sample', pd.Series([False]*len(df))).astype(str).str.lower().isin(['false', '0'])] if 'do_sample' in df.columns else df
    if len(df_greedy) == 0:
        df_greedy = df

    configs = ['Unconstrained', 'Whole-Sequence', 'Token-Aware']
    x = np.arange(len(models))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, cfg in enumerate(configs):
        vals = []
        for model in models:
            sub = df_greedy[(df_greedy['model_short'] == model) & (df_greedy['config'] == cfg)]
            vals.append(sub[col].mean() if (col in sub.columns and len(sub) > 0) else np.nan)
        bars = ax.bar(x + i * width, vals, width, label=cfg,
                      color=PROC_COLORS[cfg], alpha=0.85, edgecolor='white', linewidth=0.5)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                label_txt = f'{val:.1f}%'
                y_pos = bar.get_height() + 0.5
                ax.text(bar.get_x() + bar.get_width() / 2, y_pos, label_txt,
                        ha='center', va='bottom', fontsize=9, fontweight='bold' if val == 0 else 'normal')

    ax.axhline(0, color='black', linewidth=0.8, alpha=0.5, linestyle='--')
    ax.set_xticks(x + width)
    ax.set_xticklabels(models)
    ax.set_ylabel(col.replace('_', ' ').title().replace('Pct', '(%)'))
    ax.set_title(title, fontsize=13)
    ax.legend(title='Configuration')
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    ax.annotate('Constrained processors guarantee zero wrong-text errors',
                xy=(0.5, -0.14), xycoords='axes fraction', ha='center',
                fontsize=10, style='italic', color='#555555')
    plt.tight_layout()
    save_fig(fig, figname)

make_wrong_text_plot(
    cg, cg_models,
    'CoNLL-2003 Constrained Generation: Wrong Text Generation Rate (%)',
    'conll_cg_wrong_text_rate', col='wrong_text_rate_pct'
)


In [ ]:
make_wrong_text_plot(
    cg, cg_models,
    'CoNLL-2003 Constrained Generation: Unaligned Entity Rate (%)',
    'conll_cg_unaligned_entity_rate', col='unaligned_entity_rate_pct'
)


In [ ]:
def make_f1_comparison_plot(df, models, y_label, title, figname, ylim=80):
    df_bs1 = df[df['batch_size'] == 1].copy()
    df_greedy = df_bs1
    if 'do_sample' in df_bs1.columns:
        df_greedy = df_bs1[df_bs1['do_sample'].astype(str).str.lower().isin(['false', '0'])]
    if len(df_greedy) == 0:
        df_greedy = df_bs1

    configs = ['Unconstrained', 'Whole-Sequence', 'Token-Aware']
    x = np.arange(len(models))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, cfg in enumerate(configs):
        vals, errs = [], []
        for model in models:
            sub = df_greedy[(df_greedy['model_short'] == model) & (df_greedy['config'] == cfg)]
            vals.append(sub['main_f1_pct'].mean() if len(sub) > 0 else np.nan)
            errs.append(safe_err(sub['main_f1_std_pct'].mean()) if len(sub) > 0 else 0.0)
        ax.bar(x + i * width, vals, width, label=cfg, color=PROC_COLORS[cfg], alpha=0.85,
               yerr=errs, capsize=4, error_kw={'alpha': 0.7}, edgecolor='white', linewidth=0.5)

    ax.set_xticks(x + width)
    ax.set_xticklabels(models)
    ax.set_ylabel(y_label)
    ax.set_ylim(0, ylim)
    ax.set_title(title, fontsize=13)
    ax.legend(title='Configuration')
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    save_fig(fig, figname)

make_f1_comparison_plot(
    cg, cg_models,
    'F1 (%)',
    'CoNLL-2003 Constrained Generation: F1 Comparison at BS=1',
    'conll_cg_f1_comparison_bs1'
)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
cg_constrained = cg[cg['config'].isin(['Whole-Sequence', 'Token-Aware'])].copy()
if 'do_sample' in cg_constrained.columns:
    cg_constrained = cg_constrained[cg_constrained['do_sample'].astype(str).str.lower().isin(['false', '0'])]

line_styles = {'Whole-Sequence': '--', 'Token-Aware': '-'}

for model in cg_models:
    for cfg in ['Whole-Sequence', 'Token-Aware']:
        sub = cg_constrained[(cg_constrained['model_short'] == model) & (cg_constrained['config'] == cfg)]
        agg = sub.groupby('batch_size').agg(
            f1_mean=('main_f1_pct', 'mean'),
            f1_std=('main_f1_std_pct', 'mean')
        ).reindex(batch_sizes)
        ax.plot(batch_sizes, agg['f1_mean'].values,
                marker='o', linestyle=line_styles[cfg],
                color=MODEL_COLORS.get(model, 'gray'),
                linewidth=2, label=f'{model} ({cfg})', alpha=0.85)

ax.set_xticks(batch_sizes)
ax.set_xticklabels(['1', '5', '10', '64'])
ax.set_xlabel('Batch Size')
ax.set_ylabel('F1 (%)')
ax.set_title('CoNLL-2003: Token-Aware vs Whole-Sequence F1 across Batch Sizes', fontsize=13)
ax.legend(fontsize=8, ncol=2)
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'conll_cg_processor_comparison')


In [ ]:
cg_constrained2 = cg[cg['config'].isin(['Whole-Sequence', 'Token-Aware'])].copy()
if 'do_sample' in cg_constrained2.columns:
    cg_constrained2 = cg_constrained2[cg_constrained2['do_sample'].astype(str).str.lower().isin(['false', '0'])]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for model in cg_models:
    for cfg in ['Token-Aware']:
        sub = cg_constrained2[(cg_constrained2['model_short'] == model) & (cg_constrained2['config'] == cfg)]
        agg_wt = sub.groupby('batch_size')['wrong_text_rate_pct'].mean().reindex(batch_sizes)
        agg_f1 = sub.groupby('batch_size')['main_f1_pct'].mean().reindex(batch_sizes)
        color = MODEL_COLORS.get(model, 'gray')
        ax1.plot(batch_sizes, agg_wt.values, marker='o', color=color, linewidth=2, label=model)
        ax2.plot(batch_sizes, agg_f1.values, marker='o', color=color, linewidth=2, label=model)

ax1.set_title('Wrong Text Rate (%) — constrained', fontsize=12)
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('%')
ax1.set_xticks(batch_sizes)
ax1.set_xticklabels(['1', '5', '10', '64'])
ax1.set_ylim(-1, 5)
ax1.yaxis.grid(True, alpha=0.4)
ax1.set_axisbelow(True)
ax1.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.6)
ax1.annotate('Always 0%', xy=(0.5, 0.08), xycoords='axes fraction', ha='center',
             fontsize=11, color='green', fontweight='bold')

ax2.set_title('F1 (%) — Token-Aware constrained', fontsize=12)
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('F1 (%)')
ax2.set_xticks(batch_sizes)
ax2.set_xticklabels(['1', '5', '10', '64'])
ax2.legend()
ax2.yaxis.grid(True, alpha=0.4)
ax2.set_axisbelow(True)

plt.suptitle('CoNLL-2003 Constrained Gen: Consistency across Batch Sizes (Token-Aware, Greedy)', fontsize=13)
plt.tight_layout()
save_fig(fig, 'conll_cg_consistency_vs_batch_size')


In [ ]:
cg_ta = cg[(cg['config'] == 'Token-Aware')].copy()
if 'do_sample' in cg_ta.columns:
    cg_ta = cg_ta[cg_ta['do_sample'].astype(str).str.lower().isin(['false', '0'])]

fig, ax = plt.subplots(figsize=(9, 4))
for model in cg_models:
    sub = cg_ta[cg_ta['model_short'] == model]
    if 'elapsed_minute_avg' in sub.columns:
        agg = sub.groupby('batch_size')['elapsed_minute_avg'].mean().reindex(batch_sizes)
        ax.plot(batch_sizes, agg.values, marker='o', linewidth=2,
                color=MODEL_COLORS.get(model, 'gray'), label=model)

ax.set_xticks(batch_sizes)
ax.set_xticklabels(['1', '5', '10', '64'])
ax.set_xlabel('Batch Size')
ax.set_ylabel('Runtime (minutes)')
ax.set_title('CoNLL-2003 Constrained Generation: Runtime vs Batch Size (Token-Aware, Greedy)', fontsize=13)
ax.legend()
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'conll_cg_runtime')


## 4. CoNLL-2003: Cross-Approach Comparison

We compare the best-performing configuration for each approach: context-based with fuzzy matching at BS=5, and constrained generation (token-aware, greedy) at BS=5. GPT-OSS 20B appears only in the context-based results since it was not evaluated in the HuggingFace constrained-gen pipeline.


In [ ]:
ctx_bs5 = ctx_fuzzy[ctx_fuzzy['batch_size'] == 5]
ctx_best = ctx_bs5.groupby('model_short')['main_f1_pct'].mean().reset_index()
ctx_best.columns = ['model_short', 'f1']
ctx_best['approach'] = 'Context-Based'

cg_bs5 = cg[(cg['batch_size'] == 5) & (cg['config'] == 'Token-Aware')].copy()
if 'do_sample' in cg_bs5.columns:
    cg_bs5 = cg_bs5[cg_bs5['do_sample'].astype(str).str.lower().isin(['false', '0'])]
cg_best = cg_bs5.groupby('model_short')['main_f1_pct'].mean().reset_index()
cg_best.columns = ['model_short', 'f1']
cg_best['approach'] = 'Constrained-Gen'

combined = pd.concat([ctx_best, cg_best], ignore_index=True)
all_models = ['Gemma-3 4B', 'Qwen3 8B', 'GPT-OSS 20B', 'Llama-3.1 8B']

x = np.arange(len(all_models))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))

colors_approach = {'Context-Based': '#4E79A7', 'Constrained-Gen': '#F28E2B'}
for i, approach in enumerate(['Context-Based', 'Constrained-Gen']):
    vals = []
    for model in all_models:
        row = combined[(combined['model_short'] == model) & (combined['approach'] == approach)]
        vals.append(row['f1'].values[0] if len(row) > 0 else np.nan)
    bars = ax.bar(x + i * width, vals, width, label=approach,
                  color=colors_approach[approach], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(all_models)
ax.set_ylabel('F1 (%)')
ax.set_title('CoNLL-2003: Best F1 per Model — Context-Based vs Constrained Generation (BS=5)', fontsize=13)
ax.legend(title='Approach')
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
ax.annotate('* GPT-OSS 20B not available for constrained-gen', xy=(0.99, 0.01),
            xycoords='axes fraction', ha='right', fontsize=9, style='italic', color='#666')
plt.tight_layout()
save_fig(fig, 'conll_cross_approach_f1')


In [ ]:
rows = []
for model in all_models:
    ctx_row = combined[(combined['model_short'] == model) & (combined['approach'] == 'Context-Based')]
    cg_row = combined[(combined['model_short'] == model) & (combined['approach'] == 'Constrained-Gen')]
    ctx_f1 = f"{ctx_row['f1'].values[0]:.1f}%" if len(ctx_row) > 0 else '—'
    cg_f1 = f"{cg_row['f1'].values[0]:.1f}%" if len(cg_row) > 0 else '—'
    rows.append({'Model': model, 'Context-Based F1 (BS=5)': ctx_f1, 'Constrained-Gen F1 (BS=5)': cg_f1})

df_summary = pd.DataFrame(rows)
df_summary.style.set_caption('CoNLL-2003: Best F1 per Approach and Model').hide(axis='index')


## 5. ToxicSpans: Toxic Span Detection

ToxicSpans is a character-level span classification task. All experiments use batch size 1. We evaluate the same two approaches and report character-level F1 (char-F1).


In [ ]:
ts_ctx_exact = load_ctx(
    [BASE / 'ToxicSpans/Context-Based/Csv/toxic_spans_context.csv'], char_level=True)
ts_ctx_fuzzy = load_ctx(
    [BASE / 'ToxicSpans/Context-Based/Csv/toxic_spans_context_fuzzy.csv'], char_level=True)
ts_cg = load_cg(
    [BASE / 'ToxicSpans/Constrained-Gen/Csv/hf_all_configs_eval_1_BS_toxic_spans.csv'],
    char_level=True)

print(f"ToxicSpans context rows: {len(ts_ctx_exact)}, CG rows: {len(ts_cg)}")
ts_cg_models = [m for m in cg_models if m in ts_cg['model_short'].values]


In [ ]:
ts_models_ctx = [m for m in ['Gemma-3 4B', 'Qwen3 8B', 'GPT-OSS 20B', 'Llama-3.1 8B']
                 if m in ts_ctx_exact['model_short'].values]
prompts_ts = sorted(ts_ctx_exact['prompt_short'].dropna().unique()) if 'prompt_short' in ts_ctx_exact.columns else ['Base']
x = np.arange(len(ts_models_ctx))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, prompt in enumerate(prompts_ts[:3]):
    sub = ts_ctx_exact[ts_ctx_exact.get('prompt_short', pd.Series()) == prompt] if 'prompt_short' in ts_ctx_exact.columns else ts_ctx_exact
    vals = [sub[sub['model_short'] == m]['main_f1_pct'].mean() if len(sub[sub['model_short'] == m]) > 0 else np.nan
            for m in ts_models_ctx]
    errs = [safe_err(sub[sub['model_short'] == m]['main_f1_std_pct'].mean()) if len(sub[sub['model_short'] == m]) > 0 else 0
            for m in ts_models_ctx]
    ax.bar(x + i * width, vals, width, label=prompt,
           color=list(PROMPT_COLORS.values())[i], alpha=0.85,
           yerr=errs, capsize=4, error_kw={'alpha': 0.7})

ax.set_xticks(x + width)
ax.set_xticklabels(ts_models_ctx)
ax.set_ylabel('Char-F1 (%)')
ax.set_title('ToxicSpans Context-Based: Char-F1 by Prompt Variant', fontsize=13)
ax.legend(title='Prompt')
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'toxic_ctx_char_f1_by_prompt')


In [ ]:
make_wrong_text_plot(
    ts_cg, ts_cg_models,
    'ToxicSpans Constrained Generation: Wrong Text Generation Rate (%)',
    'toxic_cg_wrong_text_rate', col='wrong_text_rate_pct'
)


In [ ]:
make_f1_comparison_plot(
    ts_cg, ts_cg_models,
    'Char-F1 (%)',
    'ToxicSpans Constrained Generation: Char-F1 Comparison',
    'toxic_cg_char_f1_comparison', ylim=80
)


## 6. LegalQA: Extractive Legal Question Answering

The LegalQAEval dataset contains long legal documents. Spans can be multi-sentence passages, making this the hardest generalization challenge. We report character-level F1. A key anomaly: constrained generation *underperforms* unconstrained for Gemma-3 4B on this dataset — likely because the constrained processor forces character-by-character reproduction of very long legal passages, degrading span boundary precision.


In [ ]:
lq_ctx_exact = load_ctx(
    [BASE / 'LegalQAEval/Context-Based/Csv/legalqa_context_1_BATCHSZ.csv'], char_level=True)
lq_ctx_fuzzy = load_ctx(
    [BASE / 'LegalQAEval/Context-Based/Csv/legalqa_context_fuzzy_1_BATCHSZ.csv'], char_level=True)
lq_cg = load_cg(
    [BASE / 'LegalQAEval/Constrained-Gen/Csv/hf_all_configs_eval_1_BS_legalqa.csv'],
    char_level=True)

print(f"LegalQA context rows: {len(lq_ctx_exact)}, CG rows: {len(lq_cg)}")
lq_cg_models = [m for m in cg_models if m in lq_cg['model_short'].values]


In [ ]:
lq_models_ctx = [m for m in ['Gemma-3 4B', 'Qwen3 8B', 'GPT-OSS 20B', 'Llama-3.1 8B']
                 if m in lq_ctx_exact['model_short'].values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, label) in zip(axes, [(lq_ctx_exact, 'Exact'), (lq_ctx_fuzzy, 'Fuzzy')]):
    prompts_lq = sorted(df['prompt_short'].dropna().unique()) if 'prompt_short' in df.columns else ['Base']
    x = np.arange(len(lq_models_ctx))
    width = 0.25
    for i, prompt in enumerate(prompts_lq[:3]):
        sub = df[df['prompt_short'] == prompt] if 'prompt_short' in df.columns else df
        vals = [sub[sub['model_short'] == m]['main_f1_pct'].mean() if len(sub[sub['model_short'] == m]) > 0 else np.nan
                for m in lq_models_ctx]
        ax.bar(x + i * width, vals, width, label=prompt,
               color=list(PROMPT_COLORS.values())[i], alpha=0.85)
    ax.set_xticks(x + width)
    ax.set_xticklabels(lq_models_ctx)
    ax.set_ylabel('Char-F1 (%)')
    ax.set_title(f'LegalQA Context-Based ({label} Matching): Char-F1', fontsize=12)
    ax.legend(title='Prompt')
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)

plt.tight_layout()
save_fig(fig, 'legalqa_ctx_char_f1_by_prompt')


In [ ]:
make_wrong_text_plot(
    lq_cg, lq_cg_models,
    'LegalQA Constrained Generation: Wrong Text Generation Rate (%)',
    'legalqa_cg_wrong_text_rate', col='wrong_text_rate_pct'
)


In [ ]:
make_f1_comparison_plot(
    lq_cg, lq_cg_models,
    'Char-F1 (%)',
    'LegalQA Constrained Generation: Char-F1 Comparison (Anomaly: Gemma constrained < unconstrained)',
    'legalqa_cg_char_f1_comparison', ylim=70
)


## 7. Cross-Dataset Summary

Summary heatmaps showing the best F1 per (model × dataset) combination for each approach.


In [ ]:
datasets = {
    'CoNLL-2003': (ctx_fuzzy, cg),
    'ToxicSpans': (ts_ctx_fuzzy, ts_cg),
    'LegalQA': (lq_ctx_fuzzy, lq_cg),
}
all_models_ctx = ['Gemma-3 4B', 'Qwen3 8B', 'GPT-OSS 20B', 'Llama-3.1 8B']
all_models_cg = ['Gemma-3 4B', 'Qwen3 8B', 'Llama-3.1 8B']

def build_heatmap_matrix(models, ds_dict, approach):
    mat = []
    for model in models:
        row = []
        for ds_name, (ctx_df, cg_df) in ds_dict.items():
            if approach == 'context':
                sub = ctx_df[ctx_df['model_short'] == model]
                val = sub['main_f1_pct'].mean() if len(sub) > 0 else np.nan
            else:
                df_ta = cg_df[(cg_df['config'] == 'Token-Aware') & (cg_df['model_short'] == model)]
                if 'do_sample' in df_ta.columns:
                    df_ta = df_ta[df_ta['do_sample'].astype(str).str.lower().isin(['false', '0'])]
                val = df_ta['main_f1_pct'].mean() if len(df_ta) > 0 else np.nan
            row.append(val)
        mat.append(row)
    return pd.DataFrame(mat, index=models, columns=list(ds_dict.keys()))

mat_ctx = build_heatmap_matrix(all_models_ctx, datasets, 'context')
mat_cg = build_heatmap_matrix(all_models_cg, datasets, 'constrained')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (mat, title) in zip(axes, [
    (mat_ctx, 'Context-Based: Best Char/Token-F1 (%)'),
    (mat_cg, 'Constrained-Gen (Token-Aware, Greedy): F1 (%)'),
]):
    sns.heatmap(mat.astype(float), ax=ax, annot=True, fmt='.1f',
                cmap='YlGn', linewidths=0.5, linecolor='white',
                vmin=0, vmax=70, cbar_kws={'label': 'F1 (%)'})
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Model')
    ax.set_xlabel('Dataset')

plt.suptitle('Cross-Dataset Performance Summary', fontsize=14, y=1.02)
plt.tight_layout()
save_fig(fig, 'summary_heatmaps')


In [ ]:
dataset_labels = ['CoNLL-2003', 'ToxicSpans', 'LegalQA']
cg_dfs = [cg, ts_cg, lq_cg]
all_cg_models = ['Gemma-3 4B', 'Qwen3 8B', 'Llama-3.1 8B']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, (ds_label, df) in zip(axes, zip(dataset_labels, cg_dfs)):
    df_g = df
    if 'do_sample' in df.columns:
        df_g = df[df['do_sample'].astype(str).str.lower().isin(['false', '0'])]
    configs = ['Unconstrained', 'Whole-Sequence', 'Token-Aware']
    models_here = [m for m in all_cg_models if m in df_g['model_short'].values]
    x = np.arange(len(models_here))
    width = 0.25
    for i, cfg in enumerate(configs):
        vals = []
        for model in models_here:
            sub = df_g[(df_g['model_short'] == model) & (df_g['config'] == cfg)]
            vals.append(sub['wrong_text_rate_pct'].mean() if (
                'wrong_text_rate_pct' in sub.columns and len(sub) > 0) else np.nan)
        bars = ax.bar(x + i * width, vals, width, label=cfg,
                      color=PROC_COLORS[cfg], alpha=0.85, edgecolor='white')
        for bar, val in zip(bars, vals):
            if not np.isnan(val) and val == 0:
                ax.text(bar.get_x() + bar.get_width() / 2, 1,
                        '0%', ha='center', va='bottom', fontsize=8, color='green', fontweight='bold')

    ax.set_xticks(x + width)
    ax.set_xticklabels([m.replace(' ', '\n') for m in models_here], fontsize=9)
    ax.set_title(ds_label, fontsize=12, fontweight='bold')
    ax.set_ylabel('Wrong Text Rate (%)' if ax == axes[0] else '')
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    if ax == axes[2]:
        ax.legend(title='Config', fontsize=9)

plt.suptitle('Wrong Text Generation Rate — Constrained vs Unconstrained (All Datasets)',
             fontsize=13, y=1.01)
plt.tight_layout()
save_fig(fig, 'summary_wrong_text_all_datasets')


In [ ]:
summary_rows = []
for ds_label, ctx_df, cg_df_local in zip(
    dataset_labels,
    [ctx_fuzzy, ts_ctx_fuzzy, lq_ctx_fuzzy],
    [cg, ts_cg, lq_cg]
):
    for model in ['Gemma-3 4B', 'Qwen3 8B', 'GPT-OSS 20B', 'Llama-3.1 8B']:
        ctx_sub = ctx_df[ctx_df['model_short'] == model]
        ctx_f1 = f"{ctx_sub['main_f1_pct'].mean():.1f}%" if len(ctx_sub) > 0 else '—'
        cg_sub = cg_df_local[(cg_df_local['model_short'] == model) & (cg_df_local['config'] == 'Token-Aware')]
        if 'do_sample' in cg_sub.columns:
            cg_sub = cg_sub[cg_sub['do_sample'].astype(str).str.lower().isin(['false', '0'])]
        cg_f1 = f"{cg_sub['main_f1_pct'].mean():.1f}%" if len(cg_sub) > 0 else '—'
        summary_rows.append({
            'Dataset': ds_label, 'Model': model,
            'Context-Based F1': ctx_f1, 'Constrained-Gen F1 (Token-Aware)': cg_f1
        })

df_final = pd.DataFrame(summary_rows)
(df_final.style
    .set_caption('Final Performance Summary: Best F1 per Model × Dataset × Approach')
    .hide(axis='index'))
